# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Version: {metadata.get('version', 'N/A')}")
print(f"Published: {metadata.get('datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** In Croissant, entities such as record sets and fields are referenced by their `@id`. Below, we list `record_set`, `field`, and possible columns available.

In [ ]:
# List all available record sets by their @id
record_sets = []
for obj in dataset.metadata.to_json().get('recordSet', []):
    if isinstance(obj, dict) and '@id' in obj:
        record_sets.append(obj['@id'])
    elif isinstance(obj, str):
        record_sets.append(obj)

print("Record Sets (@id):")
for rset in record_sets:
    print(f"  - {rset}")

# If record sets are empty, attempt to list one by manual assignment (since Croissant for this dataset may require seed)
if not record_sets:
    # Manually set the main CSV record set @id (based on Croissant spec or `distribution` reference)
    record_sets = ['https://api.app.sen.science/frontiers/7160411/794de323-23ec-4496-8bd5-d9c5b848dafe']
    print("Using dataset record set:", record_sets[0])

# Now, print fields for the selected record set
selected_record_set = record_sets[0]
try:
    fields = dataset.fields(record_set=selected_record_set)
    print(f"Fields in RecordSet [{selected_record_set}] (by @id):")
    for field in fields:
        print(f"  - {field['@id']}: {field.get('name', field.get('@id'))}")
except Exception as e:
    print("Could not retrieve fields for record set:", selected_record_set, e)

# Show a few example records
try:
    for i, x in enumerate(dataset.records(record_set=selected_record_set)):
        print(x)
        if i > 2:
            break
except Exception as e:
    print("Could not fetch sample records:", e)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from the main record set
dataframes = {}

# For this dataset, we use the main record set that links to the survey table
main_record_set_id = record_sets[0]
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
dataframes[main_record_set_id] = df

print(f"Columns in the DataFrame from record set {main_record_set_id}:")
print(df.columns.tolist())

df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

In this example, we'll:
1. Filter records on GAD-7 score (commonly numeric, mental health indicator).
2. Normalize the GAD-7 score.
3. Group by 'village' (demographic categorical field) to explore averages.

In [ ]:
# Identify the field for GAD-7 score (search for related column; for this demonstration we use its presumed column name or @id)
# Replace as appropriate from the schema field listing
gad7_field_id = 'GAD7_score'  # Replace with actual @id if different
village_field_id = 'village'  # Replace with actual @id

if gad7_field_id not in df.columns:
    # Try common variants (case, underscore)
    candidates = [c for c in df.columns if 'gad' in c.lower()]
    if candidates:
        gad7_field_id = candidates[0]

if village_field_id not in df.columns:
    candidates = [c for c in df.columns if 'village' in c.lower()]
    if candidates:
        village_field_id = candidates[0]

print(f"Using GAD-7 field: {gad7_field_id}")
print(f"Using Village field: {village_field_id}")

# Filter records: GAD-7 score > 10
threshold = 10
filtered_df = df[df[gad7_field_id] > threshold]
print(f"Filtered records with {gad7_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize GAD-7 score
filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
print(f"Normalized {gad7_field_id} for filtered records:")
print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

# Group by village, check if field exists
if village_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(village_field_id)[gad7_field_id].mean().reset_index()
    grouped_df = grouped_df.rename(columns={gad7_field_id: f"{gad7_field_id}_mean"})
    print(f"Grouped data by {village_field_id} (mean {gad7_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of GAD-7 scores and visualize the average GAD-7 score by village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7 scores
plt.figure(figsize=(8, 5))
sns.histplot(df[gad7_field_id], bins=20, kde=True, color='skyblue')
plt.title('Distribution of GAD-7 Scores')
plt.xlabel('GAD-7 Score')
plt.ylabel('Frequency')
plt.show()

# Average GAD-7 score by village (if grouped data exists)
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 6))
    sns.barplot(x=village_field_id, y=f"{gad7_field_id}_mean", data=grouped_df)
    plt.xticks(rotation=45)
    plt.title('Average GAD-7 Score by Village')
    plt.xlabel('Village')
    plt.ylabel('Mean GAD-7 Score')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset offers comprehensive survey data from Kilifi County, Kenya, including scores from mental health screening tools and relevant demographics.
- Using `mlcroissant`, we loaded metadata, extracted records, and analyzed GAD-7 scores.
- Filtering for high GAD-7 scores (>10) provided insights into distribution among surveyed participants.
- Data normalization and grouping by village illustrated potential regional variations in mental health indicators.

This notebook can be extended to include further analyses (such as PHQ-9, PSQ scores or cohort comparisons), supporting research and community health policy or AI model development.